# Introduction

This notebook demonstrates the general workflow to process the cleaned parquets from generated from `data_cleaning.ipynb`. This notebook in particular focuses on the AmBe dataset showing neutron-gamma discrimination. 

# Setting Up

## Imports

In [ ]:
from pathlib import Path
import re
import time

from IPython.display import display
from ipywidgets import interact, interactive, interactive_output
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit, minimize_scalar
from scipy import interpolate
import scipy.signal as signal
import tomli

## Functions

### Plotting

In [ ]:
def plot_signal(
    data: pd.DataFrame, 
    sample_interval: float = 2
) -> tuple[plt.Figure, plt.Axes]:
    if data is None: return None
    
    fig, ax = plt.subplots(figsize=(15,8))
    ax.plot(
        np.arange(0, data.shape[0] * sample_interval, sample_interval), 
        data
    )

    ax.grid(visible=True)
    ax.tick_params(axis='both', labelsize=14)
    
    ax.set_xlabel("time (ns)", fontsize=18)
    ax.set_ylabel("amplitude ($V$)", fontsize=18)
    
    return fig, ax


def plot_bases(
    signal: pd.Series, 
    peak_idx: int,
    peak_height: float,
    left_idx: int,
    left_idx_user: int,
    right_idx: int,
    right_idx_user: int,
    sample_interval: float = 2
) -> tuple[plt.Figure, plt.Axes]:
    
    if signal is None:
        return None
    
    marker_offset = 0.008
    
    fig, ax = plot_signal(signal, sample_interval)

    # Plot Peak
    ax.plot(
        peak_idx * sample_interval, 
        peak_height, 
        'rx'
    )
    
    # Left Base
    ax.plot(
        left_idx * sample_interval,
        signal[left_idx] - marker_offset,
        'r^',
        label="Left Base (SciPy)"
    )
    
    
    ax.plot(
        left_idx_user * sample_interval,
        signal[left_idx_user] + marker_offset,
        'gv',
        label="Left Base (User)"
    )
    
    # Right Base
    ax.plot(
        right_idx * sample_interval,
        signal[right_idx] - marker_offset,
        'r^',
        label="Right Base (SciPy)"
    )
    
    ax.plot(
        right_idx_user * sample_interval,
        signal[right_idx_user] + marker_offset,
        'gv',
        label="Right Base (User)"
    )
    
    ax.set_title(f"Plot of Signal {signal.name}")
    ax.legend()
    
    return fig, ax

### PSD

In [ ]:
def generate_psd(
    df: pd.DataFrame, 
    left_bases: dict, 
    right_bases: dict, 
    amplitudes: dict, 
    cutoff_voltage: float
) -> pd.DataFrame:
    def process(series: pd.Series) -> dict:
        q_total = series[left_bases[series.name]:].sum()
        q_tail = series[right_bases[series.name]:].sum()
        q_peak = q_total - q_tail
        
        res = {
            "total integral": q_total,
            "tail integral": q_tail,
            "tail / total": q_tail / q_total,
        }

        return res
    
    psd_report = pd.DataFrame({signal_id: process(df[signal_id]) for signal_id in df.columns})
    psd_report.loc["amplitude"] = amplitudes
    
    filt = psd_report.loc["tail / total"].between(-0.1, 0.8)
    psd_report = psd_report.T[filt].T
    
    amp_filt = psd_report.loc["amplitude"] >= cutoff_voltage
    
    return psd_report.T[amp_filt].T


def df_to_psd(
    df: pd.DataFrame, 
    dc_offset: float,
    peak_offset: float, 
    tail_onset: float,
    cutoff_voltage: float,
) -> pd.DataFrame:
    """
    This function applies the full workflow of this notebook to the input df.
    
    Args
    -----
    peak_offset: index shifted from the peak where its base may begin; used to calculate the peak's base value
    tail_onset: index shifted from the peak where the tail should start 
    """
    df = df - dc_offset
    # df_norm = (df - df.min()) / (df.max() - df.min())
    output = df.apply(lambda x: get_bases(x, peak_idx[x.name][0], peak_offset, tail_onset))
    left_bases, right_bases = output.iloc[0,:], output.iloc[1,:]
    return generate_psd(df, left_bases, right_bases, df.max(), cutoff_voltage)

### Peak Finding

In [ ]:
def get_bases(
    series: pd.Series, 
    peak_idx: int, 
    peak_offset: int=10, 
    tail_offset=5
) -> tuple[list, list]:
    return get_left_bases(series, peak_idx, peak_offset), get_right_bases(series, peak_idx, tail_offset)


def get_left_bases(
    series: pd.Series, 
    peak_idx: int, 
    peak_offset: int=0
) -> int:
    return series[peak_idx - peak_offset: peak_idx].idxmin()


def get_right_bases(
    series: pd.Series, 
    peak_idx: int, 
    tail_offset: int=0
) -> int:
    # return series[peak_idx: peak_idx + tail_onset].idxmin()
    return peak_idx + tail_offset

### FOM

In [ ]:
def gaussian(x: list, mu: float, sigma: float, A: float) -> list:
    return A * np.exp(-((x - mu)**2 / (2 * (sigma**2))))


def bimodal(
    x: list, 
    mu1: float, 
    sigma1: float, 
    A1: float, 
    mu2: float, 
    sigma2: float, 
    A2: float
) -> list:
    return gaussian(x, mu1, sigma1, A1) + gaussian(x, mu2, sigma2, A2)


def FWHM(sigma: float) -> float:
    return 2 * np.sqrt(2 * np.log(2)) * sigma


def FOM(mu1: float, sigma1: float, mu2: float, sigma2: float) -> float:
    return abs(mu2 - mu1) / (FWHM(sigma1) + FWHM(sigma2))


def guess_bimodal_params(
    counts: np.ndarray, 
    bins: np.ndarray, 
    widths: list =[3, 5, 10, 20]
) -> float:   
    idxs = signal.find_peaks_cwt(counts, widths) # May need adjusting?
        
    if len(idxs) > 2:
        pk_idx1 = counts[idxs].argmax()
        pk_idx2 = np.delete(counts, pk_idx1).argmax()
    else:
        pk_idx1, pk_idx2 = idxs
        
    sigma1, sigma2 = 0.02, 0.02
    
    return bins[:-1][pk_idx1], sigma1, counts[pk_idx1], bins[:-1][pk_idx2], sigma2, counts[pk_idx2]


def fit_fom(
    counts: list,
    bins: list,
    n_bins: int=100,
    guesses: tuple=None
) -> tuple[tuple, float]:
    if guesses is None:
        try:
            starting_guesses = guess_bimodal_params(counts, bins)
        except ValueError:
            starting_guesses = (0.2, 0.01, 800, 0.4, 0.01, 200)
    else:
        starting_gueses = guesses
        
    params, cov = curve_fit(bimodal, bins[:-1], counts, starting_guesses)
        
    return abs(params), cov 
    

def plot_fom(
    psd: list,
    unimodal: bool = False,
    n_bins: int = 100,
    guesses: tuple = None
) -> tuple[plt.Figure, plt.Axes, dict]:    
    counts, bins = np.histogram(psd, n_bins)      
    
    params, cov = fit_fom(counts, bins, n_bins=n_bins, guesses=guesses)
    
    fig, ax = plt.subplots(figsize=(12,9))
    
    ax.plot(
        bins[:-1],
        counts,
        "--",
        label="Original PSD"
    )  
    
    if not unimodal:
        ax.plot(
            bins[:-1],
            gaussian(bins[:-1], *params[0:3]),
            label="Neutron",
            c="orange"
        )
    
    ax.plot(
        bins[:-1],
        gaussian(bins[:-1], *params[3:]),
        label="$\gamma$",
        c="indigo"
    )

    fom_val = "N/A" if unimodal else f"{FOM(params[0], params[1], params[3], params[4]):.3f}"
    ax.set_xlim(-0.1, 0.8)
    ax.set_title(f"FoM: {fom_val}")
    ax.legend()

    ax.set_ylabel("counts", fontsize=14)
    ax.set_xlabel("tail/total (a.u.)", fontsize=14)
    
    params_dict = {
        "neutron": {"mu1": params[0], "sigma1": params[1], "A1": params[2]}, 
        "gamma": {"mu2": params[3], "sigma2": params[4], "A2": params[5]}
    }
    return fig, ax, params_dict

# Pre-Processing

In [ ]:
EXP_ROOT = Path("../sample_datasets/20220906_AmBe/processed_data/cleaned_buffers/")
PARQ_PATH = EXP_ROOT / "20220906-0005_clean.parquet"

In [ ]:
df = pd.read_parquet(PARQ_PATH)
df.columns = df.columns.astype("int16")
df = df.T

## Finer DC Adjustments

This section allows us to adjust for DC offset even further if the one provided before was not suitable. Please change this value until the final result is desired. We suggest starting in step size in order of $0.001$ or less to see how it affects the data in the [Visualizations](#Visualizations) section.

In [ ]:
dc_offset = -0.00144
df_offset = df - dc_offset

In [ ]:
df_processed = df_offset

## Normalization

Currently this sections is removed as it does not do anything. In previous versions, `df_norm` was used to find peaks in [Peak Finding](#Peak-Finding), but this posed problems when dealing with noiser data; i.e. this step exaggerates the noise.

# Pulse Shape Discrimination

This section outlines the workflow for Pulse Shape Discrimination (PSD). The steps are as follows:
1. [Peak Finding](#Peak-Finding)
2. [Integration](#Integration)
3. [Visualizations](#Visualizations)
4. [Optimization](#Optimization)

More information can be found in each section's introduction.

## Peak Finding
Since we already found peaks in the `data_cleaning.ipynb` notebook, we can just reuse the settings to determine the same values! In a more complete notebook, we can just reuse these values instead of needing to recompute any information using `scipy.signal.find_peaks()`

In [ ]:
SETTINGS_PATH = EXP_ROOT / "settings.toml"

with open(SETTINGS_PATH, "rb") as f:
    exp_info = tomli.load(f)

multipeak_filter_settings = exp_info["multipeak_filter_settings"]
height = multipeak_filter_settings["height"]
prominence = multipeak_filter_settings["prominence"]

output = df_processed.apply(lambda x: signal.find_peaks(x, height=height, prominence=prominence))
peak_idx, props = output.iloc[0,:], output.iloc[1,:]

### Visualize Peak Finding

Use the interactable plot below to decide where the user `peak_offset` and `tail_onset` should lie, these are highlighted in green.

In [ ]:
random_sample = df_processed.sample(10, axis=1, random_state=42)
sample_ids = random_sample.columns   

signal_id_dropdown = widgets.Dropdown(options=sample_ids) 
tail_onset_box = widgets.BoundedIntText(value=8, min=1, max=30)
peak_offset_box = widgets.BoundedIntText(value=20, min=1, max=30)

def plot_peak_finding_interactable(
    signal_id: str, 
    peak_offset: int, 
    tail_onset: int
) -> tuple[list, list]:
   
    output = df_processed.apply(lambda x: get_bases(x, peak_idx[signal_id][0], peak_offset, tail_onset))
    left_bases, right_bases = output.iloc[0,:], output.iloc[1,:]

    plot_bases(
        df_processed.get(signal_id), 
        peak_idx[signal_id][0],
        props[signal_id]["peak_heights"],
        props[signal_id]["left_bases"],  # from scipy.signal.find_peaks()
        left_bases[signal_id],  # from us
        props[signal_id]["right_bases"],  # from scipy.signal.find_peaks()
        right_bases[signal_id],  # from us
    )

    return left_bases, right_bases

In [ ]:
peak_plot_interactable = interactive(
    lambda signal_id, peak_offset, tail_onset: plot_peak_finding_interactable(signal_id, peak_offset, tail_onset),
    signal_id = signal_id_dropdown, 
    peak_offset = peak_offset_box, 
    tail_onset = tail_onset_box
)

display(peak_plot_interactable)

## Integration

**ASSUMPTION:** We will take our `peak_offset` and `tail_onset` (instead of `scipy`'s) result for this integration.

In [ ]:
start_time = time.perf_counter()

peak_offset, tail_onset = peak_offset_box.value, tail_onset_box.value

output = df_processed.apply(lambda x: get_bases(x, peak_idx[x.name][0], peak_offset, tail_onset))
left_bases, right_bases = output.iloc[0,:], output.iloc[1,:]

psd_report = generate_psd(df_processed, left_bases, right_bases, df_processed.max(), cutoff_voltage=0.25)

print(
    f"Generated PSD report in [\x1b[1;32m{(time.perf_counter() - start_time)*1000:.2f} ms\x1b[0m]."
)

In [ ]:
psd_report

## Visualizations

Here are a couple of key figures used for pulse shape discrimination. If you would not like to run through the integration steps before running the graphs, consider using the [interactive plot](#Interactive) at the end of the notebook for a faster workflow. However, you may find it useful to use this manual section to help you create your figures for publication.

### Manual Plotting

#### Tail ($Q_\text{tail}$) vs Total ($Q_\text{total}$)

In [ ]:
fig1, ax1 = plt.subplots(figsize=(12,9))

ax1.scatter(
    psd_report.loc["total integral"],
    psd_report.loc["tail integral"],
    marker=".",
    s=1
)

ax1.set_ylim(-0.4, 8)
ax1.set_xlabel("total integral (a.u.)")
ax1.set_ylabel("tail integral (a.u.)")

plt.show()

#### $Q_\text{tail/total}$ vs Peak Amplitude

In [ ]:
fig2, ax2 = plt.subplots(figsize=(12,9))

ax2.scatter(
    psd_report.loc["amplitude"],
    psd_report.loc["tail / total"],
    marker=".",
    s=1
)

ax2.set_ylim(-0.1, 0.8)

ax2.set_xlabel("pulse amplitude ($V$)")
ax2.set_ylabel("tail / total (a.u.)")


plt.show()

In [ ]:
fig3, ax3 = plt.subplots(figsize=(12,9))

n_bins = 100  # If it doesn't show as a nice plot, try increasing resolution (hint: start at 10,000)

ax3.hist(
    psd_report.loc["tail / total"],
    bins=n_bins,
    linewidth=2,
    histtype='step',
    orientation='horizontal'
)

ax3.text(
    x=0.75,
    y=0.95,
    s=f"n_bins = {n_bins}",
    transform=ax3.transAxes
)

ax3.set_ylim(-0.1, 0.8)

ax3.set_xlabel("counts")
ax3.set_ylabel("tail / total (a.u.)")

plt.show()

#### <span style="color:#FF9900">Figure of Merit</span>

(Here's what we've been waiting for...?)

In this section we generate a plot to show the Figure of Merit (FOM) calculated using the Full Widths at Half Maximum (FWHM). These are defined as follows:

$$\text{FOM} = \dfrac{\mu_2 - \mu_1}{\text{FWHM}(\sigma_1) + \text{FWHM}(\sigma_2)}$$


$$\text{FWHM}(\sigma) = 2\sqrt{2 \ln 2} \cdot \sigma$$

where, $\mu_i$, $\sigma_i$ are the mean and standard deviation of each Gaussian distribution, respectively.

**Note:** 
* Please select the **unimodal** checkbox to describe whether there is a single or double peak. The FOM for a unimodal plot is not meaningful by definition and should not be used.
* There is a minimum that the following cutoff voltage can go. This value is adjusted by varying cutoff voltage in `generate_psd()` [here](#Integration).

In [ ]:
unimodal_checkbox = widgets.Checkbox(value=False, description="Unimodal")
cutoff_voltage_box = widgets.FloatText(value=0.38, description="Min Pk Amp")

# filt = psd_report.loc["amplitude"] >= cutoff_voltage_box.value
# pk_adj_psd = psd_report.T[filt].T

def adjustable_voltage_fom_plot(psd_report: pd.DataFrame, cutoff_voltage: float, unimodal: bool= False):
    filt = (psd_report.loc["amplitude"] >= cutoff_voltage)
    pk_adj_psd = psd_report.T[filt].T
    _, _, gauss_params = plot_fom(pk_adj_psd.loc["tail / total"], n_bins=n_bins, unimodal=unimodal)
    
    return pk_adj_psd, gauss_params

fom_interactive = interactive(
    lambda cutoff_voltage, unimodal: 
    adjustable_voltage_fom_plot(psd_report, cutoff_voltage, unimodal), 
    cutoff_voltage = cutoff_voltage_box,
    unimodal = unimodal_checkbox
)

display(fom_interactive)

pk_adj_psd = fom_interactive.result[0]

### Interactive

The following interactive plot runs the whole notebook in a condensed section. Plotting functions are kept here for cleanliness. Please resize `xlim` and `ylim` if the graphs are out of bounds. 

**NOTE:** You find it helpful to comment out where these limits are set to help you visualize the entire plot and show the outliers.

#### Functions

In [ ]:
# ALL IN ONE INTERACTIVE PLOT
def plot_psd(
    psd_report: pd.DataFrame, 
    n_bins:int = 100, 
    unimodal: bool=False,
    guesses=None
) -> tuple[plt.Figure, plt.Axes]:
    label_fs = 14
    
    fig, axs = plt.subplots(1, 3, figsize=(18,6))
    
    # Q_tail vs Q_tot
    axs[0].scatter(
        psd_report.loc["total integral"],
        psd_report.loc["tail integral"],
        marker='.',
        s=1
    )
    
    axs[0].set_ylim(-0.2, 8)
    
    axs[0].set_xlabel("total integral (a.u.)", fontsize=label_fs)
    axs[0].set_ylabel("tail integral (a.u.)", fontsize=label_fs)

    
    hist_y_lims = [-0.1, 0.8]
    
    # Q_tot vs Amplitude
    axs[1].scatter(
        psd_report.loc['amplitude'],
        psd_report.loc["tail / total"],
        marker='.',
        s=1
    )
    
    
    axs[1].set_ylim(hist_y_lims)
                    
    axs[1].set_xlabel("pulse amplitude (V)", fontsize=label_fs)
    axs[1].set_ylabel("tail / total (a.u.)", fontsize=label_fs)
    
    
    # Q distribution
    axs[2].hist(
        psd_report.loc["tail / total"],
        bins=n_bins,
        linewidth=1,
        histtype='step',
        label="psd",
        orientation='horizontal'
    )

    axs[2].text(
        x=0.05,
        y=0.95,
        s=f"n_bins = {n_bins}",
        transform=axs[2].transAxes
    )

    axs[2].set_ylim(hist_y_lims)

    axs[2].set_xlabel("counts", fontsize=label_fs)
    axs[2].set_ylabel("tail / total (a.u.)", fontsize=label_fs)
    
    
    #FOM
    counts, bins = np.histogram(psd_report.loc["tail / total"], n_bins)      
    params, cov = fit_fom(counts, bins, n_bins=n_bins, guesses=guesses)
    
    if not unimodal:
        axs[2].plot(
            gaussian(bins[:-1], *params[0:3]),
            bins[:-1],
            label="Neutron",
            c="orange"
        )

    axs[2].plot(
        gaussian(bins[:-1], *params[3:]),
        bins[:-1],
        label="$\gamma$",
        c="indigo"
    )

    fom_val = "N/A" if unimodal else f"{FOM(params[0], params[1], params[3], params[4]):.3f}"

    axs[2].text(
        x=0.05,
        y=0.90,
        s=f"FOM = {fom_val}",
        transform=axs[2].transAxes
    )
    axs[2].legend()
    
    fig.set_facecolor('white')
    fig.tight_layout()
    
    return fig, axs
    
    
def timed_plot_psd(
    df: pd.DataFrame, 
    dc_offset: float,
    peak_offset: int, 
    tail_onset: int,
    n_bins: int,
    cutoff_voltage: float,
    unimodal: bool,
) -> dict[tuple[plt.Figure, plt.Axes], pd.DataFrame, dict]:
    start_time = time.perf_counter()

    psd_report = df_to_psd(df, dc_offset, peak_offset, tail_onset, cutoff_voltage)

    fig, axs = plot_psd(psd_report, n_bins, unimodal)

    print(
        f"Action completed in [\x1b[1;32m{(time.perf_counter() - start_time)*1000:.2f} ms\x1b[0m]."
    )
    
    return_dict = {
        "plot": (fig, axs), 
        "psd_report": psd_report, 
        "configs": {
            "dc_offset": dc_offset,
            "peak_offset": peak_offset,
            "tail_onset": tail_onset,
            "cutoff_voltage": cutoff_voltage
        }
    }
    
    return return_dict

#### Plotting


**PROPOSED FEATURE:** be able to adjust xlim / ylim without replotting?

In [ ]:
tail_onset_box_end = widgets.IntText(value=8, description="Tail Onset")
peak_offset_box_end = widgets.IntText(value=20, description="Peak Offset")
dc_offset_box = widgets.FloatText(value=-0.00144, description="DC Offset")
n_bins_box = widgets.IntText(value=100, description="n_bins")
cutoff_voltage_box = widgets.FloatText(value=0.38, description="Min Pk Amp")

interactive_plot_full = interactive(
    lambda dc_offset, peak_offset, tail_onset, n_bins, cutoff_voltage, unimodal:
    timed_plot_psd(
        df, 
        dc_offset,
        peak_offset, 
        tail_onset, 
        n_bins,
        cutoff_voltage,
        unimodal
    ),
    dc_offset= dc_offset_box,
    peak_offset= peak_offset_box_end,
    tail_onset= tail_onset_box_end,
    n_bins= n_bins_box,
    cutoff_voltage= cutoff_voltage_box,
    unimodal= unimodal_checkbox
)

display(interactive_plot_full) 

## Optimization
In this section we try to find the cutoff voltage corresponding to $\text{FOM} = 1.27$ This indicates a confidence of $3\sigma$ that we characterized the gamma and neutrons correctly, noted in Lintereur _et al._ ([PNNL, 2012](https://www.pnnl.gov/main/publications/external/technical_reports/PNNL-21609.pdf)).

**NOTE:**
* **This value is not useful for unimodal plots**
* Warning may appear. This is due to fitting errors, this does not result in major problems; see the note in [Visualize FOM vs Cutoff](#Visualize-FOM-vs-Cutoff).

In [ ]:
def fom_optimize(psd_report: pd.DataFrame, min_voltage: float, n_bins: int= 100, target: float=1.27) -> float:
    filt = psd_report.loc["amplitude"]  >= min_voltage
    counts, bins = np.histogram(psd_report.loc["tail / total"].T[filt].T, n_bins)
    params, cov = fit_fom(counts, bins, n_bins)
    return abs(FOM(params[0], params[1], params[3], params[4]) - target)

In [ ]:
target = 1.27
cutoff_voltage_opt = minimize_scalar(lambda x: fom_optimize(psd_report, x, target=target), bracket=(0.1, 0.35)) 
print(f"Minimum Cutoff at {cutoff_voltage_opt.x:.4f} V for FOM = {target}.")

### Visualize FOM vs Cutoff
Here  we try to visualize the change in FOM due to varying cutoff voltages. Evidently, increasing the cutoff gives us greater certainty of neutron discrimination but we progressively lose counts in the process of doing so.

**NOTE:**
* **Results may or may not show if unimodal; in any case, it is not useful!**
* Filter used as fitting can fail at certain cutoffs resulting in random spikes (plot unfiltered to see).
* The graph is flat from the region $0\leq \text{cutoff} \leq 0.25$ because `cutoff_voltage` was set when `generate_psd()` was called in [Integration](#Integration). This will change if `cutoff_voltage` is changed there.

In [ ]:
x_fom = np.arange(0, 1, 0.05)
y_fom = []

for cutoff in x_fom:
    filt = psd_report.loc["amplitude"]  >= cutoff
    counts, bins = np.histogram(psd_report.loc["tail / total"].T[filt].T, n_bins)
    params, _ = fit_fom(counts, bins)
    y_fom.append(FOM(params[0], params[1], params[3], params[4]))

In [ ]:
y_fom = np.array(y_fom)
filt = (y_fom > 1.1) & (y_fom < 1.5)

In [ ]:
fig, ax = plt.subplots(figsize=(12,6))
ax.plot(x_fom[filt], y_fom[filt], 'o-')
ax.set_xlabel("Cutoff (V)")
ax.set_ylabel("FOM")
ax.hlines(target, 0, 1, color='k', linestyles='--', linewidth=1)

xlim_0 = 0.2
ax.text(x=xlim_0 + 0.01, y=target+0.007, s=f"FOM={target}")
ax.set_xlim(xlim_0, 0.8)

plt.show()

## Counting

In this section we attempt to count the number of neutrons that are within the counting window. 

The counting window is defined as the region: $$\mu + \sigma \leq w \leq \mu + n\sigma$$

In other words, we are taking the window spanning $n$ times the standard deviation above the mean of the gamma distribution.



### Functions

In [ ]:
def n_sigma_classifier(
    psd: pd.DataFrame, 
    n: float, 
    mu: float, sigma: float, A: float
) -> tuple[pd.DataFrame, pd.DataFrame, interpolate.interp1d, interpolate.interp1d]:
    counts, bins = np.histogram(psd.loc["tail / total"], n_bins)
    voltage_space = np.linspace(0, psd.loc["amplitude"].max() + 5)
    
    gamma_gauss = gaussian(bins[:-1], mu, sigma, A)
    gate_gauss = gaussian(bins[:-1], mu + n*sigma, sigma, A)
    
    y_gate = gate_gauss[gate_gauss.argmax():]
    y_gate_bins = bins[:-1][gate_gauss.argmax():]
    y_gamma = gamma_gauss[gamma_gauss.argmax():]
    y_gamma_bins = bins[:-1][gamma_gauss.argmax():]
    
    f_gate = interpolate.interp1d(y_gate_bins, y_gate, kind="linear", fill_value="extrapolate")
    f_gamma = interpolate.interp1d(y_gamma_bins, y_gamma, kind="linear", fill_value="extrapolate")
    
    def neutron_filter(signal):
        # amplitude and tail/total swapped because axis are swapped during graphing
        gate_cond = (signal["amplitude"] <= f_gate(signal["tail / total"]))  
        gamma_cond = (signal["amplitude"] > f_gamma(signal["tail / total"]))
        return gate_cond and gamma_cond
    
    filt = psd.apply(neutron_filter)

    neutrons = psd.T[filt].T
    gammas = psd.T[~filt].T
    
    return neutrons, gammas, f_gate, f_gamma


def plot_classification(
    neutrons: pd.DataFrame, 
    gammas: pd.DataFrame
) -> tuple[plt.Figure, plt.Axes]:
    fig, ax = plt.subplots(figsize=(12,9))
    
    ax.scatter(
        neutrons.loc["amplitude"],
        neutrons.loc["tail / total"],
        marker='s',
        s=8,
        color='k',
        label="Neutron"
    )

    ax.scatter(
        gammas.loc["amplitude"],
        gammas.loc["tail / total"],
        marker='.',
        s=8,
        color='indigo',
        label="$\gamma$"
    )

    ax.set_xlabel("pulse amplitude ($V$)", fontsize=14)
    ax.set_ylabel("tail / total (a.u.)", fontsize=14)
    ax.set_xlim(0, 2.5)
    ax.set_ylim(-0, 0.5)
    ax.legend()
    
    return fig, ax


def interactive_plot_classification(psd_report, cutoff_voltage: float, n: float=8):
    filt = (psd_report.loc["amplitude"] >= cutoff_voltage)
    pk_adj_psd = psd_report.T[filt].T
    gamma_fit_params = fom_interactive.result[1]["gamma"]
    neutrons, gammas, f_gate, f_gamma = n_sigma_classifier(pk_adj_psd, n, *gamma_fit_params.values())

    fig, ax = plot_classification(neutrons, gammas)
    ax.plot(
    f_gate(voltage_space),
    voltage_space,
    "r--"
    )

    ax.plot(
        f_gamma(voltage_space),
        voltage_space,
        "r--"
    )

    ax.text(
        1.73,
        0.48,
        s=f"n_neutron = {neutrons.shape[1]}"
    )

    ax.text(
        1.73,
        0.46,
        s=f"n_gamma = {gammas.shape[1]}"
    )

    ax.fill_betweenx(voltage_space, f_gamma(voltage_space), f_gate(voltage_space), alpha=0.1, color="orange")
    ax.vlines(cutoff_voltage, 0, 1, color="k", linestyles="--", linewidth=1.2)
    ax.set_title(f"Neutron Classification at n={n} and cutoff = {cutoff_voltage:.3f}V")
    fig.tight_layout()

### Plotting

In [ ]:
n_sigma_box = widgets.FloatText(value=8, description="n")  
cutoff_voltage_class_box = widgets.FloatText(value=0.38, description="Min Pk Amp")

classification_interactive = interactive(
    lambda cutoff_voltage, n: interactive_plot_classification(psd_report, cutoff_voltage, n),
    cutoff_voltage = cutoff_voltage_class_box,
    n = n_sigma_box
)

display(classification_interactive)

### Neutrons over Time

In [ ]:
EXP_TIMES_PATH = "../sample_datasets/20220906_AmBe/raw_data/exp_times.csv"

buffer_number = int(re.split("-(\d+)[_.]", PARQ_PATH.name)[1])
elapsed_time = pd.read_csv(EXP_TIMES_PATH, index_col=0).at[buffer_number, "elapsed_s"]

counts_per_second = neutrons.shape[1] / elapsed_time
print(f"At n={n} and cutoff voltage={cutoff_voltage_box.value}V:\x1b[1;31m {counts_per_second:.3f} Neutrons/s \x1b[0m")

#